# ONNX Inference Examples

This notebook demonstrates how to use the ONNX models for inference in various scenarios.


In [ ]:
import json
import time
from pathlib import Path
from typing import  Tuple

import numpy as np
import pandas as pd
import onnxruntime as rt



In [ ]:
class ONNXPredictor:
    """Simple predictor class using ONNX models."""

    def __init__(self, model_dir: Path = Path("onnx_models")):
        """
        Initialize predictor with ONNX models.

        Args:
            model_dir: Directory containing ONNX models
        """
        self.model_dir = Path(model_dir)

        # Load scaler
        scaler_path = self.model_dir / "scaler.onnx"
        self.scaler_sess = rt.InferenceSession(str(scaler_path))
        self.scaler_input_name = self.scaler_sess.get_inputs()[0].name
        self.scaler_output_name = self.scaler_sess.get_outputs()[0].name

        # Load model
        model_path = self.model_dir / "model.onnx"
        self.model_sess = rt.InferenceSession(str(model_path))
        self.model_input_name = self.model_sess.get_inputs()[0].name

        # Load metadata
        metadata_path = self.model_dir / "onnx_metadata.json"
        with open(metadata_path, 'r') as f:
            self.metadata = json.load(f)

        self.n_features = self.metadata['n_features']
        self.feature_names = self.metadata['feature_names']

        print("ONNX Predictor initialized")
        print(f"   Features: {self.n_features}")
        print(f"   Model: {self.metadata['original_model'][:50]}...")

    def predict(
        self,
        X: np.ndarray,
        return_probabilities: bool = True
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        Make predictions on input data.

        Args:
            X: Input features, shape (n_samples, n_features)
            return_probabilities: Whether to return class probabilities

        Returns:
            Tuple of (predictions, probabilities)
            - predictions: Class labels (0=Ocean, 1=Land)
            - probabilities: Class probabilities, shape (n_samples, 2)
        """
        # Validate input
        if X.shape[1] != self.n_features:
            raise ValueError(
                f"Expected {self.n_features} features, got {X.shape[1]}"
            )

        # Ensure float32 dtype
        X = X.astype(np.float32)

        # Step 1: Scale input
        X_scaled = self.scaler_sess.run(
            [self.scaler_output_name],
            {self.scaler_input_name: X}
        )[0]

        # Step 2: Predict
        outputs = self.model_sess.run(None, {self.model_input_name: X_scaled})

        predictions = outputs[0]
        probabilities = outputs[1] if len(outputs) > 1 else None

        if return_probabilities and probabilities is not None:
            return predictions, probabilities
        else:
            return predictions, None

    def predict_single(self, features: np.ndarray) -> dict:
        """
        Predict on a single sample with detailed output.

        Args:
            features: Feature vector, shape (n_features,) or (1, n_features)

        Returns:
            Dictionary with prediction results
        """
        # Reshape if needed
        if features.ndim == 1:
            features = features.reshape(1, -1)

        # Predict
        predictions, probabilities = self.predict(features)

        result = {
            'prediction': int(predictions[0]),
            'prediction_label': 'Land' if predictions[0] == 1 else 'Ocean',
            'confidence': float(np.max(probabilities[0])) if probabilities is not None else None,
            'probabilities': {
                'Ocean': float(probabilities[0][0]) if probabilities is not None else None,
                'Land': float(probabilities[0][1]) if probabilities is not None else None
            }
        }

        return result

print("ONNXPredictor class defined")

ONNXPredictor class defined


In [24]:
# Initialize predictor
predictor = ONNXPredictor()

ONNX Predictor initialized
   Features: 78
   Model: xgboost_compressed_with_features_290925_1025_deliv...


## Real Test Data

In [25]:
print("\n" + "="*70)
print("Prediction with Real Test Data")
print("="*70)

# Load a small sample from real test data
processed_data_dir = Path("../processed_data")
test_features_path = processed_data_dir / "raw_counts_test_features.parquet"

if test_features_path.exists():
    print("\nLoading real test data...")
    X_test = pd.read_parquet(test_features_path)
    
    # Use first 10 samples
    X_sample = X_test.iloc[:10].values.astype(np.float32)
    
    print(f"Loaded {len(X_sample)} samples")

    # Predict
    predictions, probabilities = predictor.predict(X_sample)
    
    print("\nReal Data Predictions:")
    print(f"\n{'Sample':<10} {'Prediction':<15} {'Confidence':<15} {'Ocean Prob':<15} {'Land Prob':<15}")
    print("-" * 70)
    
    for i in range(len(X_sample)):
        label = "Land" if predictions[i] == 1 else "Ocean"
        conf = probabilities[i][int(predictions[i])]
        p_ocean = probabilities[i][0]
        p_land = probabilities[i][1]
        
        print(f"{i+1:<10} {label:<15} {conf:<15.4f} {p_ocean:<15.4f} {p_land:<15.4f}")
else:
    print(f"\nTest data not found at: {test_features_path}")


Prediction with Real Test Data

Loading real test data...
Loaded 10 samples

Real Data Predictions:

Sample     Prediction      Confidence      Ocean Prob      Land Prob      
----------------------------------------------------------------------
1          Ocean           0.8406          0.8406          0.1594         
2          Land            0.9511          0.0489          0.9511         
3          Ocean           0.9205          0.9205          0.0795         
4          Ocean           0.9534          0.9534          0.0466         
5          Land            0.9780          0.0220          0.9780         
6          Ocean           0.6843          0.6843          0.3157         
7          Ocean           0.9916          0.9916          0.0084         
8          Ocean           0.8902          0.8902          0.1098         
9          Land            0.6231          0.3769          0.6231         
10         Ocean           0.9935          0.9935          0.0065         


## Performance Benchmark

In [26]:
print("\n" + "="*70)
print("Performance Benchmark")
print("="*70)

batch_sizes = [1, 10, 100, 1000]

print("\n Benchmarking inference speed...\n")
print(f"{'Batch Size':<15} {'Time (ms)':<15} {'Throughput':<20}")
print("-" * 50)

for batch_size in batch_sizes:
    X = np.random.randn(batch_size, 78).astype(np.float32)

    # Warm-up
    predictor.predict(X)

    # Benchmark
    start = time.time()
    n_runs = 100 if batch_size <= 100 else 10
    for _ in range(n_runs):
        predictor.predict(X)
    elapsed = time.time() - start

    avg_time_ms = (elapsed / n_runs) * 1000
    throughput = batch_size * n_runs / elapsed

    print(f"{batch_size:<15} {avg_time_ms:<15.2f} {throughput:>10,.0f} samples/s")


Performance Benchmark

 Benchmarking inference speed...

Batch Size      Time (ms)       Throughput          
--------------------------------------------------
1               0.07                14,300 samples/s
10              1.57                 6,351 samples/s
100             2.67                37,452 samples/s
1000            20.75               48,190 samples/s


## Saving Predictions